# Day 10 - Visual EDA

This notebook performs read-only exploratory analysis of image quality, annotation/class distribution, scene coverage proxies, and degradation patterns. It uses the Day 7 raw archive and writes derived grids, plots, JSON summaries, and hypotheses to `eda_outputs/`.

The notebook does not resize, enhance, relabel, split, or delete source data. If the archive is empty, it reports that state and creates no fabricated statistics.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import json
import math
import re

try:
    from PIL import Image, ImageDraw, ImageStat
except ImportError:
    Image = ImageDraw = ImageStat = None

try:
    import matplotlib.pyplot as plt
    import numpy as np
except ImportError:
    plt = np = None

NOTEBOOK_DIR = Path.cwd()
ARCHIVE_CANDIDATES = [
    NOTEBOOK_DIR.parent / 'day7_hemamdhar_initialinspection' / 'raw_data_archive',
    NOTEBOOK_DIR / 'raw_data_archive',
]
ARCHIVE = next((path for path in ARCHIVE_CANDIDATES if path.exists()), ARCHIVE_CANDIDATES[0])
OUTPUTS = NOTEBOOK_DIR / 'eda_outputs'
OUTPUTS.mkdir(exist_ok=True)
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
LABEL_EXTENSIONS = {'.txt'}
print('Notebook directory:', NOTEBOOK_DIR)
print('Archive:', ARCHIVE)
print('Archive exists:', ARCHIVE.exists())
print('Pillow available:', Image is not None)
print('Plotting available:', plt is not None and np is not None)

In [ ]:
def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def class_names_for(dataset_root):
    candidates = list(dataset_root.rglob('data.yaml')) + list(dataset_root.rglob('data.yml'))
    for config in candidates:
        names = {}
        for line in config.read_text(encoding='utf-8', errors='replace').splitlines():
            match = re.match(r'\s*(?:names\s*:\s*)?(\d+)\s*:\s*[\"\]?(.+?)[\"\]?\s*$', line)
            if match:
                names[int(match.group(1))] = match.group(2).strip(" \"'")
        if names:
            return names
    return {}

def parse_yolo(path, class_names):
    rows, errors = [], []
    try:
        lines = path.read_text(encoding='utf-8').splitlines()
    except Exception as exc:
        return rows, [f'read_error:{type(exc).__name__}']
    for line_number, line in enumerate(lines, 1):
        parts = line.split()
        if not parts:
            continue
        if len(parts) != 5:
            errors.append(f'line_{line_number}:expected_5_fields')
            continue
        try:
            class_id = int(parts[0])
            values = [float(value) for value in parts[1:]]
        except ValueError:
            errors.append(f'line_{line_number}:non_numeric')
            continue
        if class_id not in class_names and class_names:
            errors.append(f'line_{line_number}:unknown_class_{class_id}')
        if any(value < 0 or value > 1 for value in values):
            errors.append(f'line_{line_number}:coordinate_out_of_range')
        if values[2] <= 0 or values[3] <= 0:
            errors.append(f'line_{line_number}:non_positive_box_size')
        rows.append((class_id, *values))
    return rows, errors

def image_for_label(label_path):
    for candidate in label_path.parent.parent.glob(f'{label_path.stem}.*'):
        if candidate.suffix.lower() in IMAGE_EXTENSIONS:
            return candidate
    return None

def safe_percentile(values, percentile):
    if not values:
        return None
    return float(np.percentile(values, percentile)) if np is not None else sorted(values)[min(len(values) - 1, int(len(values) * percentile / 100))]

In [ ]:
image_records = []
label_records = []
class_counts = Counter()
class_image_counts = Counter()
error_counts = Counter()
dataset_roots = sorted(path for path in ARCHIVE.iterdir() if path.is_dir()) if ARCHIVE.exists() else []

for dataset_root in dataset_roots:
    dataset = dataset_root.name
    class_names = class_names_for(dataset_root)
    for image_path in sorted(dataset_root.rglob('*')):
        if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        record = {'dataset': dataset, 'path': str(image_path), 'relative_path': str(image_path.relative_to(dataset_root)), 'bytes': image_path.stat().st_size}
        if Image is not None:
            try:
                with Image.open(image_path) as image:
                    rgb = image.convert('RGB')
                    stat = ImageStat.Stat(rgb)
                    pixels = list(rgb.resize((1, 1)).getdata())[0]
                    record.update({
                        'format': image.format, 'width': image.width, 'height': image.height, 'mode': image.mode,
                        'aspect_ratio': round(image.width / image.height, 5) if image.height else None,
                        'mean_rgb': [round(value, 3) for value in stat.mean],
                        'mean_brightness': round(sum(stat.mean) / 3 / 255, 5),
                        'brightness_std': round(sum(stat.rms) / 3 / 255, 5),
                        'thumbnail_rgb': [int(value) for value in pixels],
                    })
            except Exception as exc:
                record['read_error'] = type(exc).__name__
        image_records.append(record)

    for label_path in sorted(dataset_root.rglob('labels/*.txt')):
        rows, errors = parse_yolo(label_path, class_names)
        image_path = image_for_label(label_path)
        for row in rows:
            class_counts[(dataset, row[0])] += 1
        for class_id in {row[0] for row in rows}:
            class_image_counts[(dataset, class_id)] += 1
        for error in errors:
            error_counts[(dataset, error.split(':', 1)[-1])] += 1
        label_records.append({
            'dataset': dataset, 'label': str(label_path), 'image': str(image_path) if image_path else None,
            'objects': len(rows), 'errors': errors, 'class_ids': sorted({row[0] for row in rows}),
        })

print('Datasets:', [root.name for root in dataset_roots])
print('Images inspected:', len(image_records))
print('Labels inspected:', len(label_records))
print('Class instances:', dict(class_counts))
print('Label errors:', dict(error_counts))

In [ ]:
def summarize(values):
    numeric = [value for value in values if isinstance(value, (int, float)) and math.isfinite(value)]
    return {
        'count': len(numeric),
        'min': min(numeric) if numeric else None,
        'p25': safe_percentile(numeric, 25),
        'median': safe_percentile(numeric, 50),
        'p75': safe_percentile(numeric, 75),
        'max': max(numeric) if numeric else None,
    }

quality_summary = {}
for dataset in sorted({record['dataset'] for record in image_records}):
    records = [record for record in image_records if record['dataset'] == dataset]
    quality_summary[dataset] = {
        'images': len(records),
        'read_errors': sum('read_error' in record for record in records),
        'width': summarize([record.get('width') for record in records]),
        'height': summarize([record.get('height') for record in records]),
        'aspect_ratio': summarize([record.get('aspect_ratio') for record in records]),
        'brightness': summarize([record.get('mean_brightness') for record in records]),
        'brightness_std': summarize([record.get('brightness_std') for record in records]),
        'classes': {str(class_id): {'instances': class_counts[(dataset, class_id)], 'images': class_image_counts[(dataset, class_id)]} for (_, class_id) in class_counts if _ == dataset},
    }

summary = {
    'archive': str(ARCHIVE),
    'archive_exists': ARCHIVE.exists(),
    'image_count': len(image_records),
    'label_count': len(label_records),
    'quality_by_dataset': quality_summary,
    'label_errors': {f'{dataset}|{error}': count for (dataset, error), count in error_counts.items()},
}
(OUTPUTS / 'visual_eda_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))

In [ ]:
# Plot image-quality distributions when there are readable images and matplotlib is available.
if plt is None or not image_records:
    print('Quality plots skipped: plotting dependency or image records are unavailable.')
else:
    datasets = sorted({record['dataset'] for record in image_records})
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    for dataset in datasets:
        records = [record for record in image_records if record['dataset'] == dataset]
        axes[0].hist([record.get('width') for record in records if record.get('width')], alpha=0.55, label=dataset)
        axes[1].hist([record.get('height') for record in records if record.get('height')], alpha=0.55, label=dataset)
        axes[2].hist([record.get('mean_brightness') for record in records if record.get('mean_brightness') is not None], alpha=0.55, label=dataset)
    axes[0].set_title('Image width'); axes[1].set_title('Image height'); axes[2].set_title('Mean brightness')
    for axis in axes:
        axis.set_ylabel('Images'); axis.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(OUTPUTS / 'image_quality_distributions.png', dpi=160)
    plt.show()

In [ ]:
# Plot class-instance and class-image support.
if plt is None or not class_counts:
    print('Class plot skipped: plotting dependency or parsed labels are unavailable.')
else:
    labels = [f'{dataset} | class {class_id}' for dataset, class_id in sorted(class_counts)]
    values = [class_counts[key] for key in sorted(class_counts)]
    fig, axis = plt.subplots(figsize=(12, max(4, len(labels) * 0.28)))
    axis.barh(labels, values, color='#176b87')
    axis.set_title('Annotated object instances by dataset and class')
    axis.set_xlabel('Object instances')
    fig.tight_layout()
    fig.savefig(OUTPUTS / 'class_instance_distribution.png', dpi=160)
    plt.show()

In [ ]:
def draw_grid(records, output_path, title, max_items=12):
    if Image is None or plt is None or not records:
        print(f'Grid skipped for {title}: dependency or records unavailable.')
        return
    selected = records[:max_items]
    columns = 4
    rows = math.ceil(len(selected) / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(16, rows * 4))
    axes = list(axes.flat) if hasattr(axes, 'flat') else [axes]
    for axis, record in zip(axes, selected):
        try:
            image = Image.open(record['path']).convert('RGB')
            axis.imshow(image)
            axis.set_title(f"{record['dataset']}\n{Path(record['path']).name}", fontsize=8)
        except Exception as exc:
            axis.set_title(f'read error: {type(exc).__name__}', fontsize=8)
        axis.axis('off')
    for axis in axes[len(selected):]:
        axis.axis('off')
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(output_path, dpi=160)
    plt.show()

for dataset in sorted({record['dataset'] for record in image_records}):
    records = [record for record in image_records if record['dataset'] == dataset]
    draw_grid(records, OUTPUTS / f'raw_grid_{dataset}.png', f'Raw image grid - {dataset}')

difficult_records = sorted(
    [record for record in image_records if record.get('mean_brightness') is not None],
    key=lambda record: record.get('mean_brightness', 1),
)
draw_grid(difficult_records, OUTPUTS / 'darkest_sample_grid.png', 'Candidate difficult samples: darkest images')

In [ ]:
hypotheses = [
    {
        'hypothesis': 'Low brightness and high brightness variation may indicate illumination loss or strong underwater color/contrast degradation.',
        'test': 'Review the darkest and highest-variation samples against boxes and class recall; do not exclude on brightness alone.',
        'status': 'candidate; requires visual review',
    },
    {
        'hypothesis': 'Extreme aspect ratios or small resolutions may reduce localization quality for small objects.',
        'test': 'Compare resolution/aspect-ratio bins with box area and per-class support after labels are parsed.',
        'status': 'candidate; requires measured labels',
    },
    {
        'hypothesis': 'Adjacent extracted frames may create near-duplicate leakage across train, validation, and test.',
        'test': 'Use hashes and filename/source-sequence groups before freezing splits.',
        'status': 'high-priority data-control hypothesis',
    },
    {
        'hypothesis': 'Different exports may represent different class semantics and scene domains.',
        'test': 'Review YAML class maps and representative grids separately before any class merge.',
        'status': 'high-priority data-control hypothesis',
    },
]
(OUTPUTS / 'hypotheses.json').write_text(json.dumps(hypotheses, indent=2), encoding='utf-8')
hypothesis_text = '# Visual EDA Hypotheses\n\n' + '\n'.join(f"- **{item['hypothesis']}** Test: {item['test']} Status: {item['status']}." for item in hypotheses) + '\n'
(OUTPUTS / 'hypotheses.md').write_text(hypothesis_text, encoding='utf-8')
print(hypothesis_text)

## Interpretation and next actions

1. Review `visual_eda_summary.json` for measured counts, quality distributions, readable-file failures, class support, and label errors.
2. Review each raw grid and `darkest_sample_grid.png` as candidate evidence, then record manifest IDs and dispositions in the Day 9 sample board.
3. Treat brightness, resolution, aspect ratio, and color statistics as hypotheses about degradation, not automatic exclusion rules.
4. Complete source-group duplicate checks before freezing train/validation/test manifests.
5. Update Day 9 and Day 8 reports with measured results and rerun the approval gates.